<h1>Data Validation and Exploratory Data Analysis</h1>
<hr/>
<p>This notebook focuses on understanding the dataset before model training. Similar to the original notebook, the objective is to inspect the data, verify its structure, check for missing values and duplicates, and understand whether the features have meaningful relationships with the target variable.</p>


<hr/>
<h1>1.&nbsp;&nbsp;&nbsp;&nbsp;Importing of Modules</h1>
<hr/>
<p>In order to conduct data analysis, several Python libraries are required. <code>pandas</code> is used for tabular data manipulation, while the project package provides reusable functions for loading, validating, and statistically testing the data.</p>


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


In [ ]:
import pandas as pd

from us_housing_price_prediction.data import load_housing_data
from us_housing_price_prediction.statistics import run_feature_significance_tests

pd.set_option("display.float_format", "{:.3f}".format)


<hr/>
<h1>2.&nbsp;&nbsp;&nbsp;&nbsp;Importing of Dataset</h1>
<hr/>
<p>Before any data analysis can be conducted, the dataset must be imported into a dataframe. The project now uses a canonical CSV file located at <code>data/raw/housing_price_data.csv</code>. The original file had an <code>.xls</code> extension, but the contents were actually CSV-formatted text. Renaming it improves clarity and prevents confusion.</p>


In [ ]:
df = load_housing_data()
print("Dataset successfully imported and validated")
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")
df.head(10)


<p>From the displayed dataframe, the dataset contains both numerical and categorical variables. The target variable is <code>price_usd</code>. The <code>house_id</code> column is an identifier and should not be treated as a model feature because it does not describe the physical or market characteristics of a house.</p>


<hr/>
<h1>3.&nbsp;&nbsp;&nbsp;&nbsp;Characteristics of Data</h1>
<hr/>
<p>The next step is to inspect the data types, number of rows, and numerical summary statistics. This allows us to check whether columns have been loaded correctly and whether the values appear reasonable before feature engineering and model training.</p>


In [ ]:
df.info()


In [ ]:
df.describe(include="all").T


<p>With reference to the information and description tables above, the numerical columns are stored as numerical data types and the categorical columns are stored as object columns. This is suitable because categorical encoding should happen later inside the machine learning pipeline rather than before splitting the data.</p>


<hr/>
<h1>4.&nbsp;&nbsp;&nbsp;&nbsp;Checking for Missing and Duplicated Data</h1>
<hr/>
<p>Missing values and duplicated records can affect model training and evaluation. Therefore, they must be checked before modeling. The validation function already performs these checks, but the summary below makes the results visible inside the notebook.</p>


In [ ]:
quality_summary = pd.DataFrame(
    {
        "missing_values": df.isna().sum(),
        "unique_values": df.nunique(),
    }
)
quality_summary.loc["duplicated_rows", "missing_values"] = df.duplicated().sum()
quality_summary


<p>With reference to the quality summary, the workflow can proceed if there are no unexpected missing values and no duplicated rows. This helps ensure that the model is trained on clean and consistent data.</p>


<hr/>
<h1>5.&nbsp;&nbsp;&nbsp;&nbsp;Categorical Data Analysis</h1>
<hr/>
<p>The dataset contains categorical variables such as city and renovation status. These variables must be understood because they can influence housing prices and must be encoded before model training.</p>


In [ ]:
city_counts = df["city"].value_counts().sort_index()
renovation_counts = df["renovation_status"].value_counts().sort_index()

pd.DataFrame(
    {
        "city_count": city_counts,
        "renovation_status_count": renovation_counts,
    }
)


<p>The categorical summaries show the distribution of houses across cities and renovation statuses. These categories are kept as raw categorical values at this stage because the production model uses a pipeline to encode them safely during training.</p>


<hr/>
<h1>6.&nbsp;&nbsp;&nbsp;&nbsp;Feature Significance Testing</h1>
<hr/>
<p>To support interpretation, p-value based tests are conducted. Numerical features are tested against price using Pearson correlation, while categorical features are tested using one-way ANOVA. This helps identify whether a feature has a statistically detectable relationship with housing price in this dataset.</p>
<br/>
<p>These tests do not prove that a feature causes the price to change. They only provide evidence of association within the available data.</p>


In [ ]:
significance_tests = run_feature_significance_tests(df)
significance_tests


<p>From the p-value test table, features with p-values below 0.05 are considered statistically significant at the 5% level. This provides additional support for including them in the modeling workflow, while still remembering that machine learning evaluation must be based on validation performance and not statistical tests alone.</p>
